In [1]:
# VENDOR QUALITY & LOGISTICS SCORECARD — DATA PREP & EDA
# Pipeline: raw CSVs -> clean -> feature engineering -> MySQL

In [2]:
import pandas as pd

In [5]:
#LOAD — import all 5 raw datasets
orders        = pd.read_csv("orders.csv")
order_items   = pd.read_csv("order_items.csv")
order_reviews = pd.read_csv("order_reviews.csv")
products      = pd.read_csv("products.csv")
sellers       = pd.read_csv("sellers.csv")

In [6]:
# DATA QUALITY AUDIT (before cleaning)
# For every table: shape, data types, and missing-value counts.
# This tells us WHAT needs cleaning before we touch anything.
# Findings: products has nulls in category & dimensions;
# orders has nulls in delivery dates (non-delivered orders);
# reviews has heavy nulls in comment title/message.

def audit(df, name):
    print(f"\n--- {name} ---")
    print("shape:", df.shape)
    print("information", df.info())
    missing = df.isnull().sum()
    print("missing values")
    print(missing)

for name, df in [
    ("orders", orders),
    ("order_items", order_items),
    ("order_reviews", order_reviews),
    ("products", products),
    ("sellers", sellers),]: audit(df, name)


--- orders ---
shape: (99441, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB
information None
missing values
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_

In [7]:
# CLEAN PRODUCTS — handle missing values
# Missing category -> 'Unknown' (keeps rows usable for analysis);
# missing lengths/photos/dimensions -> 0 (preserves numeric dtype
# so the columns stay usable for freight/size analysis).
products['product_category_name'] = products['product_category_name'].fillna('Unknown')

cols_to_fill = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty',
                'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in cols_to_fill:
    products[col] = products[col].fillna(0)

In [8]:
# CLEAN ORDERS — scope to real deliveries
# Keep only order_status = 'delivered': delivery-time metrics are
# meaningless for canceled/processing orders. Drop rows missing
# key dates, then convert all 5 date columns to datetime so we
# can do date arithmetic.

orders = orders[orders['order_status'] == 'delivered']
orders = orders.dropna(subset=['order_approved_at', 'order_delivered_carrier_date',
                               'order_delivered_customer_date'])

date_columns = ['order_purchase_timestamp', 'order_approved_at',
                'order_delivered_carrier_date', 'order_delivered_customer_date',
                'order_estimated_delivery_date']
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

In [9]:
# FEATURE ENGINEERING — the two core logistics metrics
# delivery_time_days = purchase -> customer door (total speed)
# sla_delay_days     = actual vs promised date
#                      (positive = LATE, negative = EARLY)
# These two columns power the entire dashboard (late %, risk score).
# ------------------------------------------------------------
orders['delivery_time_days'] = (orders['order_delivered_customer_date']- orders['order_purchase_timestamp']).dt.days
orders['sla_delay_days'] = (orders['order_delivered_customer_date']- orders['order_estimated_delivery_date']).dt.days

In [10]:
# CLEAN ORDER_ITEMS — type conversion only
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

In [11]:
# CLEAN REVIEWS — nulls, types, and DEDUPLICATION
# Text nulls -> 'NA' (only the numeric score matters for scoring).
# Dedup: 547 orders have 2+ review records (customer re-reviewed);
# keep only the LATEST review per order so one customer can't be
# counted twice in a seller's average rating.
# ------------------------------------------------------------
order_reviews['review_comment_title']   = order_reviews['review_comment_title'].fillna('NA')
order_reviews['review_comment_message'] = order_reviews['review_comment_message'].fillna('NA')
order_reviews['review_creation_date']   = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

order_reviews = (order_reviews.sort_values(['review_creation_date', 'review_answer_timestamp']).drop_duplicates(subset='order_id', keep='last'))

In [12]:
# REFERENTIAL INTEGRITY — keep child tables consistent
# After filtering orders to delivered only, drop order_items and
# reviews whose parent order no longer exists (prevents orphan
# rows and foreign-key failures when loading to MySQL).
# ------------------------------------------------------------
order_items   = order_items[order_items['order_id'].isin(orders['order_id'])]
order_reviews = order_reviews[order_reviews['order_id'].isin(orders['order_id'])]

In [13]:
# POST-CLEANING AUDIT — verify shapes/dtypes after cleaning
# (same checks as Step 2; confirms nulls handled & row counts align)
# ------------------------------------------------------------
def audit1(df, name):
    print(f"\n--- {name} ---")
    print("shape:", df.shape)
    print("information", df.info())

for name, df in [
    ("orders", orders),
    ("order_items", order_items),
    ("order_reviews", order_reviews),
    ("products", products),
    ("sellers", sellers),]: audit1(df, name)


--- orders ---
shape: (96455, 10)
<class 'pandas.core.frame.DataFrame'>
Index: 96455 entries, 0 to 99440
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96455 non-null  object        
 1   customer_id                    96455 non-null  object        
 2   order_status                   96455 non-null  object        
 3   order_purchase_timestamp       96455 non-null  datetime64[ns]
 4   order_approved_at              96455 non-null  datetime64[ns]
 5   order_delivered_carrier_date   96455 non-null  datetime64[ns]
 6   order_delivered_customer_date  96455 non-null  datetime64[ns]
 7   order_estimated_delivery_date  96455 non-null  datetime64[ns]
 8   delivery_time_days             96455 non-null  int64         
 9   sla_delay_days                 96455 non-null  int64         
dtypes: datetime64[ns](5), int64(2), object(3)
memory usa